# Instance segmentation

This notebook compares the instance segmentation and area quantification of infected cells using Ultralytics YOLOv26-seg.

## Requirements

Python 3.9 with `numpy`, `ultralytics`, `torch` and `opencv-python`. The image files (e.g., `Carioca_3841`) and model `cells/Inf_cell_YOLOv26L-seg_vers2.pt` must be located in a `cells/` folder next to this notebook.


In [2]:
# Standard Python stack
import os
import csv
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO


# -----------------------------------------------------------------------------
# 1. Reproducibility
# -----------------------------------------------------------------------------
# This pipeline performs inference only (no training, no augmentation),
# so random seeds do not affect the model outputs directly. However,
# we still fix all seeds and enable deterministic CUDA operations to
# maximise reproducibility of the reported measurements.
def set_reproducibility(seed: int = 42) -> None:
    """Fix random seeds and enable deterministic behaviour for PyTorch/CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:  # older torch versions
        print(f"torch.use_deterministic_algorithms not applied: {exc}")


set_reproducibility(42)


# -----------------------------------------------------------------------------
# 2. Configuration
# -----------------------------------------------------------------------------
# Paths and file names are intentionally left as in the original notebook.
DATA_DIR = Path('cells')

MODEL_PATH = DATA_DIR / 'Inf_cell_YOLOv26L-seg_vers2.pt'
INPUT_DIR = DATA_DIR / 'Carioca_3841'  # dir with images
OUTPUT_CSV = DATA_DIR / 'results.csv'

IMAGE_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.tif')
CONFIDENCE_THRESHOLD = 0.6  # empirically chosen


# -----------------------------------------------------------------------------
# 3. Load model and collect image files
# -----------------------------------------------------------------------------
model = YOLO(str(MODEL_PATH))

image_files = []
for ext in IMAGE_EXTENSIONS:
    image_files.extend(INPUT_DIR.glob(ext))

# Sort to ensure deterministic processing order across file systems.
image_files = sorted(image_files)
print(f"Images found for processing: {len(image_files)}")


# -----------------------------------------------------------------------------
# 4. Interactive input for the 'File Name' column
# -----------------------------------------------------------------------------
print("=" * 30)
CUSTOM_NAME = input(
    "Enter the value for the 'File Name' column (applied to all rows): "
).strip()
print("=" * 30 + "\n")


# -----------------------------------------------------------------------------
# 5. Segmentation and area quantification
# -----------------------------------------------------------------------------
# For each image, YOLOv26-seg inference is run and a union mask of all detected
# instances is built. The union-mask area is computed as the number of
# foreground pixels (mask.sum()); this avoids the approximation introduced
# by polygon filling and correctly handles overlapping instances.
with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'File Name',
        'Total Pixels',
        'Mask Pixels',
        'Coverage Percentage',
        'Objects Detected',
    ])

    for img_path in image_files:
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Failed to load file (skipped): {img_path.name}")
                continue

            h, w = img.shape[:2]
            total_pixels = h * w

            # Run segmentation.
            results = model(
                str(img_path),
                verbose=False,
                conf=CONFIDENCE_THRESHOLD,
                retina_masks=True,
            )

            # Union mask of all detected instances.
            # result.masks.data has shape (N, H, W) and contains binary masks (0/1).
            union_mask = np.zeros((h, w), dtype=np.uint8)
            object_count = 0

            for result in results:
                if result.masks is not None:
                    object_count += len(result.masks.data)
                    for i in range(len(result.masks.data)):
                        instance_mask = (
                            result.masks.data[i].cpu().numpy().astype(np.uint8)
                        )
                        union_mask = np.logical_or(
                            union_mask, instance_mask
                        ).astype(np.uint8)

            mask_pixels = int(union_mask.sum())
            coverage_pct = (
                (mask_pixels / total_pixels) * 100 if total_pixels > 0 else 0.0
            )

            # Fall back to the original file name if no custom name is provided.
            record_name = CUSTOM_NAME if CUSTOM_NAME else img_path.name

            writer.writerow([
                record_name,
                total_pixels,
                mask_pixels,
                round(coverage_pct, 2),
                object_count,
            ])
            print(
                f"Processed {img_path.name} | Objects: {object_count} | "
                f"Coverage: {coverage_pct:.2f}%"
            )

        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")


# -----------------------------------------------------------------------------
# 6. Save results (rename CSV)
# -----------------------------------------------------------------------------
print("\n" + "=" * 30)
user_filename = input("Enter the desired CSV file name (without .csv): ").strip()

if user_filename:
    final_csv = DATA_DIR / f"{user_filename}.csv"
    OUTPUT_CSV.rename(final_csv)
    print(f"Processing complete. Results saved to: {final_csv.name}")
else:
    print(f"Processing complete. No name entered. File saved as: {OUTPUT_CSV.name}")

Images found for processing: 139


Enter the value for the 'File Name' column (applied to all rows):  Carioca+3841



Processed 1.jpg | Objects: 13 | Coverage: 30.06%
Processed 10.jpg | Objects: 23 | Coverage: 36.47%
Processed 100.jpg | Objects: 40 | Coverage: 67.19%
Processed 101.jpg | Objects: 45 | Coverage: 63.15%
Processed 102.jpg | Objects: 31 | Coverage: 55.59%
Processed 103.jpg | Objects: 33 | Coverage: 54.09%
Processed 104.jpg | Objects: 33 | Coverage: 54.09%
Processed 105.jpg | Objects: 31 | Coverage: 55.59%
Processed 106.jpg | Objects: 28 | Coverage: 50.03%
Processed 107.jpg | Objects: 32 | Coverage: 48.44%
Processed 108.jpg | Objects: 27 | Coverage: 41.73%
Processed 109.jpg | Objects: 32 | Coverage: 57.52%
Processed 11.jpg | Objects: 14 | Coverage: 41.88%
Processed 110.jpg | Objects: 29 | Coverage: 53.70%
Processed 111.jpg | Objects: 31 | Coverage: 44.52%
Processed 113.jpg | Objects: 31 | Coverage: 47.29%
Processed 114.jpg | Objects: 19 | Coverage: 30.19%
Processed 115.jpg | Objects: 18 | Coverage: 38.60%
Processed 116.jpg | Objects: 17 | Coverage: 48.02%
Processed 117.jpg | Objects: 20 | 

Enter the desired CSV file name (without .csv):  data_for_3841


NameError: name 'OUTPUT_DIR' is not defined

---

*Created by Evgenii A. Kirichek 
<br>Notebook automatically reviewed and commented by Claude Sonnet 4.5*

---